# 📰 Fake News Detector
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/fake-news-detector/training/notebook.ipynb)

Reads a news **headline + article** and estimates whether it is **fake** or **real**. Four recurrent networks
(SimpleRNN, LSTM, GRU, bidirectional LSTM) are trained on the same cleaned
[ISOT Fake and Real News](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset) data; the one with
the best validation accuracy is exported to the Hugging Face model folder that the
[Space](https://huggingface.co/spaces/shalev396/fake-news-detector) serves.

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "fake-news-detector"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has; nothing platform-specific is installed or special-cased.
import tensorflow as tf
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print("device:", device)

In [ ]:
# 1d. Project code: src/ = training steps, ../model/model.py = cleaning, vectorizer, architectures, Predictor
import time

import pandas as pd
from IPython.display import Image as ShowImage, display

from src import data_setup, engine, export, model_builder, utils   # importing src puts ../model on sys.path
from src.config import Config
import model as M                                                  # ../model/model.py

print(utils.lib_versions("tensorflow", "keras", "numpy", "pandas", "sklearn", "nltk"))
print("device:", utils.device_name("tensorflow"))

## 2. Config
Every hyperparameter lives in `src/config.py`: 20,000 balanced articles, a 20,000-word vocabulary, 300 tokens per
article, 100-d embeddings, 64 recurrent units, Adam lr 1e-3, batch 64, up to 4 epochs with early stopping (patience 2).
`SMOKE_TEST=1` shrinks the run to 800 articles, a 5,000-word vocabulary, 100 tokens, 16 units and 1 epoch per
variant, and exports to `outputs/smoke/model` instead of `../model`.

In [ ]:
os.environ.setdefault("SMOKE_TEST", "0")   # "1" = quick smoke run; a SMOKE_TEST set outside the notebook wins
PUSH_TO_HUB = False                         # upload the exported model folder to the Hub (section 8)

cfg = Config()
utils.set_seeds(cfg.seed, "tensorflow")
print("export to:", cfg.model_dir)
cfg

## 3. Data
3a downloads `Fake.csv` + `True.csv` once into `training/data/` (kagglehub; the Hugging Face mirror
`GonzaloA/fake_news` is the fallback). Only `title` and `text` are kept: the `subject` and `date` columns
separate the classes on their own.

In [ ]:
# 3a. Download (cached) -> raw rows, label 0 = real, 1 = fake
raw, source = data_setup.load_raw(cfg)
print(source, f"{len(raw):,} articles")
raw.groupby("label").size().rename(index={0: "real", 1: "fake"})

3b cleans the text with the same functions the Space runs (`model.combine` + `model.clean_text`). Every real
article starts with a `CITY (Reuters) -` dateline; left in, that alone would solve the task, so it is removed
together with every `reuters` token. Then: lowercase, no URLs / digits / punctuation, no NLTK stopwords.

In [ ]:
# 3b. Deduplicate -> balanced subsample -> clean
stop_words = data_setup.get_stopwords()
df = data_setup.prepare(cfg, raw, stop_words)
row = df.iloc[0]
print("raw    :", (row.title + " | " + row.text)[:300])
print("cleaned:", row.text_clean[:300])
lengths = df["text_clean"].str.split().str.len()
print(f"{len(df):,} articles, tokens per article: median {lengths.median():.0f}, "
      f"{(lengths > cfg.sequence_length).mean():.0%} longer than sequence_length={cfg.sequence_length}")

In [ ]:
# 3c. Stratified train / val / test split, vectorizer adapted on TRAIN only
splits = data_setup.split(cfg, df)
vectorizer = data_setup.build_vectorizer(cfg, splits.x_train)
vocab = vectorizer.get_vocabulary()
X_train, X_val, X_test = (M.vectorize(vectorizer, x) for x in (splits.x_train, splits.x_val, splits.x_test))
print(splits.counts(), "| vocabulary:", len(vocab), "| X_train:", X_train.shape)

## 4. Load model
Each variant is `model.build_model(variant)`: `Embedding(mask_zero)` -> recurrent layer -> `Dropout` ->
`Dense(relu)` -> `Dense(1, sigmoid)` = P(fake). Only the recurrent layer differs between experiments.

In [ ]:
pd.DataFrame({M.VARIANTS[v]: {"architecture": model_builder.describe(v, cfg),
                               "parameters": f"{model_builder.build(v, cfg).count_params():,}"}
              for v in cfg.variants}).T

## 5. Training
Every variant starts from the same seed and is trained with early stopping on the validation loss (best weights
restored), then scored on the validation split (selection) and the test split (report).
**With `SMOKE_TEST=0` this is the full run**: about 8 minutes on a desktop CPU for all four variants
(measured by the earlier run); on a GPU it should be a few minutes (estimate, not measured).

In [ ]:
evals, trained = {}, {}
start = time.perf_counter()
for name in cfg.variants:
    print(f"--- {M.VARIANTS[name]}")
    utils.set_seeds(cfg.seed, "tensorflow")                 # identical start for every variant
    net = engine.compile_model(model_builder.build(name, cfg), cfg)
    history = engine.train(net, X_train, splits.y_train, X_val, splits.y_val, cfg)
    evals[name] = engine.evaluate(name, net, X_val, splits.y_val, X_test, splits.y_test, history)
    trained[name] = net
train_time_s = time.perf_counter() - start
print(f"trained {len(evals)} variants in {train_time_s:.0f} s")

In [ ]:
curves = cfg.outputs_dir / "training_curves.png"
export.plot_training_curves(evals, curves)
display(ShowImage(filename=str(curves)))

## 6. Evaluation
The variant with the highest **validation** accuracy is deployed; the test split is only reported.

In [ ]:
best = engine.select_best(evals)
table = pd.DataFrame({M.VARIANTS[n]: {"val_accuracy": e.val["accuracy"], **{f"test_{k}": v for k, v in e.test.items()},
                                      "params": e.params, "epochs": e.epochs_ran} for n, e in evals.items()}).T
print("deployed variant:", M.VARIANTS[best])
table.sort_values("val_accuracy", ascending=False).round(4)

In [ ]:
ranked = sorted(evals.values(), key=lambda e: (e.val["accuracy"], e.val["roc_auc"]), reverse=True)
plots = {"model_comparison": lambda p: export.plot_comparison(ranked, best, p),
         "confusion_matrix": lambda p: export.plot_confusion(engine.confusion(splits.y_test, evals[best].test_prob), best, p),
         "roc_curve": lambda p: export.plot_roc(evals, best, splits.y_test, p)}
for name, draw in plots.items():
    draw(cfg.outputs_dir / f"{name}.png")
    display(ShowImage(filename=str(cfg.outputs_dir / f"{name}.png"), width=560))

## 7. Inference
7a saves the chosen model the way the Space loads it (`model.keras`, `vocab.json`, `stopwords.json`, `config.json`)
and checks that reloading reproduces the training predictions. 7b runs `model.load(dir, device).predict(text)` on
`../space/examples`, exactly what the Space and the endpoint run: headline on the first line, article below.

In [ ]:
# 7a. Save the runtime files of the chosen model (+ round-trip check through model.py)
export.save_model(cfg, best=best, model=trained[best], vocab=vocab, stop_words=stop_words, source=source,
                  evals=evals, splits=splits)

In [ ]:
# 7b. Predict like the Space
predictor = M.load(cfg.model_dir, "cuda" if device == "GPU" else "cpu")
rows = []
for path in sorted((utils.SPACE_DIR / "examples").glob("*.txt")):
    probs = predictor.predict(path.read_text(encoding="utf-8"))
    rows.append({"example": path.name, "fake": round(probs["fake"], 4), "real": round(probs["real"], 4),
                 "prediction": max(probs, key=probs.get)})
pd.DataFrame(rows)

## 8. Export
Writes `metrics.json`, the plots in `assets/` and refreshes the model card (`README.md`: YAML `model-index`, the
metrics table and the experiments table), so `cfg.model_dir` is a complete Hugging Face model repo. To publish,
set `PUSH_TO_HUB = True` in section 2. The token comes from `hf auth login` (local) or an `HF_TOKEN`
secret / environment variable (Colab: the Secrets panel). Smoke runs are never uploaded.

In [ ]:
metrics = export.write_report(cfg, best=best, evals=evals, splits=splits, source=source,
                             device=device.lower(), train_time_s=train_time_s)
print(metrics["primary_metric"], metrics["source"])
for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{path.relative_to(cfg.model_dir).as_posix():30s} {path.stat().st_size / 1e6:8.2f} MB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))